<p style="text-align:center">
    <a href="https://tukkalearn.vercel.app" target="_blank">
    <img src="https://raw.githubusercontent.com/itzDM/publicAssets/refs/heads/main/opengraph-image.png" width="250"  alt="Tukka Learn">
    </a>
</p>


In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv("https://raw.githubusercontent.com/tukkaLearn/datasets/refs/heads/main/sales_nov2025_full.csv")
df.head()

,order_id,customer_id,order_date,product_name,category,price,quantity,total_amount,discount,country,status
0,ORD1001,CUST901,2025-11-01,iPhone 16 Pro,Electronics,1299.0,1,1299.0,0.1,USA,Delivered
1,ORD1002,CUST902,2025-11-02,AirPods,Electronics,249.0,2,498.0,0.0,Canada,Pending
2,ORD1003,CUST901,2025-11-15,MacBook Pro,Electronics,2399.0,1,2159.1,0.1,USA,Delivered
3,ORD1004,CUST215,2025-11-28,Sneakers,Fashion,120.0,5,600.0,0.2,India,Shipped
4,ORD1005,CUST440,2025-11-30,Kindle,Electronics,139.0,1,139.0,0.0,Germany,Cancelled


### Exercise 1: Which country made us the most money?


In [15]:
df.groupby('country')['total_amount'].sum().sort_values(ascending=False).head(5).round(0)

country
USA          4747.0
India         600.0
Australia     499.0
Canada        498.0
Germany       139.0
Name: total_amount, dtype: float64

### Exercise 2: Multiple metrics per country


In [3]:
country_report = df.groupby('country').agg({
    'total_amount': 'sum',
    'order_id': 'count',
    'price': 'mean',
    'discount': 'median'
}).round(2)

country_report.columns = ['Total Revenue', 'Order Count', 'Avg Price', 'Median Discount']
country_report = country_report.sort_values('Total Revenue', ascending=False)
country_report.head(8)

,Total Revenue,Order Count,Avg Price,Median Discount
country,,,,
USA,4747.1,4,1206.75,0.1
India,600.0,1,120.00,0.2
Australia,499.0,1,499.00,0.0
Canada,498.0,1,249.00,0.0
Germany,139.0,1,139.00,0.0


### Exercise 3: Custom column names with named aggregation


In [4]:
pretty = df.groupby('category', as_index=False).agg(
    revenue=('total_amount', 'sum'),
    orders=('order_id', 'nunique'),
    avg_price=('price', 'mean'),
    top_product=('product_name', lambda x: x.value_counts().index[0])
).round(2)

pretty['revenue_millions'] = (pretty['revenue'] / 1e6).round(2)
pretty.sort_values('revenue', ascending=False)

,category,revenue,orders,avg_price,top_product,revenue_millions
0,Electronics,5084.2,5,1037.0,iPhone 16 Pro,0.01
1,Fashion,899.9,2,75.0,Sneakers,0.00
2,Gaming,499.0,1,499.0,PS5,0.00


### Exercise 4: Category + Country matrix


In [5]:
matrix = df.groupby(['category', 'country'])['total_amount'].sum().unstack(fill_value=0)
matrix

country,Australia,Canada,Germany,India,USA
category,,,,,
Electronics,0.0,498.0,139.0,0.0,4447.2
Fashion,0.0,0.0,0.0,600.0,299.9
Gaming,499.0,0.0,0.0,0.0,0.0


### Exercise 5: Add country average to every row


In [ ]:
df['country_avg'] = df.groupby('country')['total_amount'].transform('mean')
df['revenue_vs_country_avg'] = df['total_amount'] / df['country_avg']

print("Orders that beat their country average by 1x+")
df[df['revenue_vs_country_avg'] > 1][['country', 'total_amount', 'revenue_vs_country_avg']].head()

Orders that beat their country average by 5x+


,country,total_amount,revenue_vs_country_avg
0,USA,1299.0,1.094563
2,USA,2159.1,1.819300


### Exercise 6: filter() – keep only active categories


In [18]:
active = df.groupby('category').filter(
    lambda x: len(x) > 1 and x['price'].mean() > 5
)

print(f"Kept {active['category'].nunique()} categories out of {df['category'].nunique()}")
active['category'].value_counts()

Kept 2 categories out of 3


category
Electronics    5
Fashion        2
Name: count, dtype: int64

### Exercise 7: Daily revenue trend


In [30]:
df["order_date"] = pd.to_datetime(df["order_date"])
df.groupby('order_date',as_index=False).agg(
      revenue=('total_amount', 'sum'),
    orders=('order_id', 'nunique'),
    avg_price=('price', 'mean'),
)

,order_date,revenue,orders,avg_price
0,2025-11-01,1299.0,1,1299.00
1,2025-11-02,498.0,1,249.00
2,2025-11-15,2159.1,1,2399.00
3,2025-11-20,299.9,1,29.99
4,2025-11-25,499.0,1,499.00
5,2025-11-28,600.0,1,120.00
6,2025-11-29,989.1,1,1099.00
7,2025-11-30,139.0,1,139.00


### Exercise 8: Top customer in each country


In [31]:
idx = df.groupby('country')['total_amount'].idxmax()
top_customers = df.loc[idx, ['country', 'customer_id', 'total_amount']].sort_values('total_amount', ascending=False)
top_customers

,country,customer_id,total_amount
2,USA,CUST901,2159.1
3,India,CUST215,600.0
6,Australia,CUST555,499.0
1,Canada,CUST902,498.0
4,Germany,CUST440,139.0


### Exercise 12: % of premium orders per category


In [26]:
premium_rate = (
    df.assign(is_premium = df['price'] > 1000)
      .groupby('category')['is_premium']
      .mean()
      .sort_values(ascending=False) * 100
).round(1)

print("Categories with >30% premium orders:")
premium_rate[premium_rate > 30]

Categories with >30% premium orders:


category
Electronics    60.0
Name: is_premium, dtype: float64

### Exercise 13: Pivot-style table without pivot_table()


In [27]:
df.groupby(['country', 'category'])['total_amount'].sum().unstack().fillna(0).style.format('{:,.0f')

AttributeError: The '.style' accessor requires jinja2

### Exercise 14: Weekly revenue (resample pro move)


In [28]:
weekly = df.set_index('date')['total_amount'].resample('W-MON').sum()
weekly.plot(kind='bar', figsize=(12,5), title='Weekly Revenue')
plt.xticks(rotation=45)
plt.show()
print("Black Friday week:", weekly['2025-11-24':'2025-11-30'].sum().round(0))

KeyError: "None of ['date'] are in the columns"

### Exercise 15: FINAL BOSS – Executive Dashboard in 10 lines


In [29]:
def exec_dashboard(df):
    print("EXECUTIVE DASHBOARD – NOV 2025")
    print("="*60)
    print(f"Total Revenue     : ${df['total_amount'].sum():,.0f}")
    print(f"Total Orders      : {len(df):,}")
    print(f"Avg Order Value   : ${df['total_amount'].mean():.2f}")
    print("\nTop 5 Countries:")
    display(df.groupby('country')['total_amount'].sum().nlargest(5).round(0))
    print("\nTop 3 Categories per Top Country:")
    top_country = df.groupby('country')['total_amount'].sum().idxmax()
    display(df[df['country']==top_country].groupby('category')['total_amount'].sum().nlargest(3))
    print(f"\nReturning customers : {df['customer_id'].duplicated().sum():,} ({df['customer_id'].duplicated().mean()*100:.1f}%)")

exec_dashboard(df)

EXECUTIVE DASHBOARD – NOV 2025
Total Revenue     : $6,483
Total Orders      : 8
Avg Order Value   : $810.39

Top 5 Countries:


country
USA          4747.0
India         600.0
Australia     499.0
Canada        498.0
Germany       139.0
Name: total_amount, dtype: float64


Top 3 Categories per Top Country:


category
Electronics    4447.2
Fashion         299.9
Name: total_amount, dtype: float64


Returning customers : 3 (37.5%)


## YOU ARE NOW OWN GROUPBY

You can answer any business question in seconds.
You are dangerous in meetings.

**Next elite notebook options:**

- `give me datetime notebook` ← (everyone’s favorite)
- `give me merge & concat notebook`
- `give me full 10-notebook zip + datasets`

Just say **"next"** or pick one.

You're in the top 0.1% now. Keep climbing!


<hr>
<div style="text-align:center">
  <h3 style="color:orange">|| राम नाम सत्य है ||</h3>
  <h4>Authour : सीता राम जी </h4>
   <h5 style="color:skyblue"><i>© All Rights Reserved</i></h5>
</div>
